# DX 704 Week 13 Project

In this project, you will use a language model API to turn unstructured text into data, then build a simple AI agent that analyzes the resulting dataset and writes a report.


You will work with fictional city service requests. Each request contains a resident's description of a problem, along with information recorded by the city's intake system. The intake classifications are imperfect: an apparently routine request might describe a significant safety hazard, and different residents may describe similar problems in very different ways.

You will complete the following task:
Give a language model access to data-analysis tools so it can investigate the enriched dataset and prepare a quantitative report.
You may use AI coding assistants (including ChatGPT, Codex, or Claude Code) to help write and debug your code, but you are responsible for verifying the outputs and conclusions.

The full project description and a template notebook are available on GitHub: [Project 13 Materials](https://github.com/bu-cds-dx704/dx704-project-13).


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

### Documentation

The following OpenAI API documentation will be helpful.

- [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Responses API](https://developers.openai.com/api/reference/resources/responses/methods/create)
- [Function Calling](https://developers.openai.com/api/docs/guides/function-calling)

### Modules

In [ ]:
%pip install -q -U duckdb matplotlib openai pydantic

In [ ]:
import datetime
import decimal
import hashlib
import json
import math
import os
import pathlib
import re
import time
import typing
import uuid

import duckdb
import matplotlib.pyplot as plt
import openai
import pandas as pd
import pydantic
import tqdm.notebook
import IPython

### Raw Data

You are given the following files:

| File | Contents |
|---|---|
| `requests.csv` | Approximately 2,000 service requests with `request_id`, `created_at`, `closed_at`, `district_id`, `description`, `intake_category`, `intake_priority`, `status`, and other operational fields |
| `districts.csv` | District names and populations |
| `events.csv` | Weather and operational events that might help explain request patterns |

In [ ]:
!wget -nc https://github.com/bu-cds-dx704/dx704-project-13/raw/main/requests.csv
!head requests.csv

In [ ]:
!wget -nc https://github.com/bu-cds-dx704/dx704-project-13/raw/main/districts.csv
!head districts.csv

In [ ]:
!wget -nc https://github.com/bu-cds-dx704/dx704-project-13/raw/main/events.csv
!head events.csv

# Part 0: OpenAI Structured Outputs Example

Review the code below. It demonstrates how an OpenAI model can convert a  free-text description into fields with specified types and allowed values.



### API Access

You should use a secure method to provide your API key (for example, environment variables or Colab secrets). Do not put API keys directly in your submitted notebook.

In [ ]:
# OpenAI API key
#
# if not using Colab, comment out these lines.

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
client = openai.OpenAI()  # Reads OPENAI_API_KEY from the environment
MODEL = "gpt-6-luna"  # You may choose another supported OpenAI model.

### Structured Response Queries

In [ ]:
class IncidentExtraction(pydantic.BaseModel):
    incident_type: typing.Literal[
        "road", "electrical", "water", "drainage",
        "sanitation", "tree", "traffic_signal", "other", "unknown"
    ]
    urgency: typing.Literal["routine", "urgent", "unknown"]
    safety_hazard: typing.Literal["yes", "no", "unknown"]
    evidence: str

response = client.responses.parse(
    model=MODEL,
    input=[
        {
            "role": "system",
            "content": (
                "Classify the described incident using only the evidence in the text. "
                "Use 'unknown' if there is not enough information. "
                "The evidence field should briefly identify text supporting the labels."
            ),
        },
        {
            "role": "user",
            "content": "A truck hit the traffic light. It is hanging over the road.",
        },
    ],
    text_format=IncidentExtraction,
)

print(response.output_parsed)

The model produces a structured object whose fields conform to the supplied schema. That does **not** guarantee the classifications are correct; you will evaluate them in Part 2.

# Part 1: Build a Data-Analysis Agent Using Tools

You will now build an agent that can analyze your enriched dataset.
This agent will decide **which data-analysis operations to perform** in response to a question.

The agent will be provided the following functions:
1. `describe_database`
2. `run_sql`
3. `create_chart`
4. `create_report`

You are given an outline of an agent, `run_analyst` that can call those functions.
Your task is to complete the integration between `run_analyst` and the functions above.


### Load Data into DuckDB

In [ ]:
DB_FILE = "city_services.duckdb"

# Setup connection: allowed to write.
setup = duckdb.connect(DB_FILE)

setup.execute("""
    CREATE OR REPLACE TABLE requests AS
    SELECT * FROM read_csv_auto('requests.csv')
""")

setup.execute("""
    CREATE OR REPLACE TABLE districts AS
    SELECT * FROM read_csv_auto('districts.csv')
""")

setup.execute("""
    CREATE OR REPLACE TABLE events AS
    SELECT * FROM read_csv_auto('events.csv')
""")

setup.close()

### Setup Read-Only Data Connection

In [ ]:
con = duckdb.connect(
    DB_FILE,
    read_only=True,
    config={
        "enable_external_access": "false",
        "autoload_known_extensions": "false",
        "autoinstall_known_extensions": "false",
        "allow_community_extensions": "false",
        "allow_unsigned_extensions": "false",
        "lock_configuration": "true",
        "memory_limit": "512MB",
        "threads": "2",
    },
)

### Tool: describe_database

In [ ]:
def describe_database(table: str | None = None) -> dict:
    """List DuckDB tables or describe a specific table."""
    try:
        # Discover available tables.
        tables = con.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
              AND table_type IN ('BASE TABLE', 'VIEW')
            ORDER BY table_name
        """).fetchall()

        table_names = [row[0] for row in tables]

        if table is not None and table not in table_names:
            return {
                "ok": False,
                "error": f"Unknown table: {table}",
                "available_tables": table_names,
            }

        def row_count(name):
            # Names are checked against the database catalog.
            quoted = '"' + name.replace('"', '""') + '"'
            return con.execute(
                f"SELECT COUNT(*) FROM {quoted}"
            ).fetchone()[0]

        # No table specified: list all tables.
        if table is None:
            return {
                "ok": True,
                "tables": [
                    {
                        "name": name,
                        "row_count": row_count(name),
                    }
                    for name in table_names
                ],
            }

        # Describe the specified table.
        columns = con.execute("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'main'
              AND table_name = ?
            ORDER BY ordinal_position
        """, [table]).fetchall()

        return {
            "ok": True,
            "table": table,
            "row_count": row_count(table),
            "columns": [
                {"name": name, "type": dtype}
                for name, dtype in columns
            ],
        }

    except Exception as e:
        return {"ok": False, "error": str(e)}


### Tool: run_sql

In [ ]:
MAX_SQL_ROWS = 100

def _execute_select(query, max_rows):
    """
    Execute one SELECT statement against DuckDB.

    Returns (columns, rows, truncated).

    Raises an exception for invalid SQL or
    disallowed statement types.
    """
    if not isinstance(query, str) or not query.strip():
        raise ValueError("SQL query must be a nonempty string.")

    statements = con.extract_statements(query)

    if (len(statements) != 1 or
        statements[0].type != duckdb.StatementType.SELECT):
        raise ValueError("Only one SELECT statement is allowed.")

    result = con.execute(query)
    columns = [col[0] for col in result.description]

    if len(set(columns)) != len(columns):
        raise ValueError(
            "Duplicate column names. Use SQL aliases."
        )

    # Fetch one extra row to detect truncation.
    rows = result.fetchmany(max_rows + 1)
    truncated = len(rows) > max_rows

    return columns, rows[:max_rows], truncated

def _json_value(value):
    """Convert common DuckDB values to JSON-safe values."""
    if isinstance(value, (datetime.date, datetime.datetime, datetime.time)):
        return value.isoformat()
    if isinstance(value, decimal.Decimal):
        return float(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, (list, tuple)):
        return [_json_value(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _json_value(v) for k, v in value.items()}
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)

def run_sql(query: str) -> dict:
    """
    Execute a read-only DuckDB SELECT query.

    Returns up to 100 rows as JSON-compatible data.
    """
    try:
        columns, rows, truncated = _execute_select(
            query, MAX_SQL_ROWS
        )

        return {
            "ok": True,
            "columns": columns,
            "rows": [
                {
                    col: _json_value(value)
                    for col, value in zip(columns, row)
                }
                for row in rows
            ],
            "row_count": len(rows),
            "truncated": truncated,
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
        }

### Tool: create_chart

In [ ]:
CHART_DIR = pathlib.Path("charts")
CHART_DIR.mkdir(exist_ok=True)

MAX_CHART_POINTS = 500

def create_chart(
    query: str,
    filename: str,
    chart_type: str = "plot",
    fmt: str = "-o",
    title: str = "",
    xlabel: str = "",
    ylabel: str = "",
    rotation: int = 0,
) -> dict:
    """
    Execute SQL returning columns x and y, then plot.

    chart_type:
        "plot" -> matplotlib.pyplot.plot(x, y, fmt)
        "bar"  -> matplotlib.pyplot.bar(x, y)

    Saves the result as a PNG in the charts directory.
    """
    fig = None

    try:
        if chart_type not in ("plot", "bar"):
            raise ValueError(
                "chart_type must be 'plot' or 'bar'."
            )

        if not isinstance(filename, str) or not re.fullmatch(
            r"[A-Za-z0-9][A-Za-z0-9_.-]*\.png", filename
        ):
            raise ValueError(
                "filename must be a simple PNG filename."
            )

        if not -90 <= rotation <= 90:
            raise ValueError(
                "rotation must be between -90 and 90."
            )

        columns, rows, truncated = _execute_select(
            query, MAX_CHART_POINTS
        )

        if columns != ["x", "y"]:
            raise ValueError(
                "Query must return exactly two columns "
                "named x and y."
            )

        if truncated:
            raise ValueError(
                "Too many points. Aggregate or filter "
                "the query to at most 500 rows."
            )

        if not rows:
            raise ValueError("Query returned no data.")

        x = [row[0] for row in rows]
        y = [row[1] for row in rows]

        if any(v is None for v in x):
            raise ValueError("x contains NULL values.")

        if any(
            isinstance(v, bool)
            or not isinstance(v, (int, float, decimal.Decimal))
            or not math.isfinite(v)
            for v in y
        ):
            raise ValueError(
                "y must contain finite numeric values."
            )

        # Thin wrapper around Matplotlib.
        fig = plt.figure(figsize=(8, 5))

        if chart_type == "plot":
            plt.plot(x, y, fmt)
        else:
            plt.bar(x, y)

        plt.title(title)
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.xticks(rotation=rotation)
        plt.tight_layout()

        path = CHART_DIR / filename
        plt.savefig(path)

        return {
            "ok": True,
            "filename": str(path),
            "chart_type": chart_type,
            "points_plotted": len(rows),
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
        }

    finally:
        if fig is not None:
            plt.close(fig)

### Tool: create_report

In [ ]:
def create_report(filename: str, text: str) -> dict:
    """Write Markdown text to a new or overwritten .md file."""
    try:
        if "/" in filename or "\\" in filename:
            raise ValueError("Filename must not contain slashes.")

        if not filename.endswith(".md"):
            raise ValueError("Filename must end in '.md'.")

        with open(filename, "w", encoding="utf-8") as f:
            f.write(text)

        return {
            "ok": True,
            "filename": filename
        }

    except (ValueError, OSError) as e:
        return {
            "ok": False,
            "error": str(e)
        }

### Analyst Agent

The `tools` list provides tool definitions to pass to the API.
The definition for the `describe_database` tool has been provided.
Fill in the tool definitions for the remaining tools defined above.
Note that your descriptions here will influence how and when the agent decides to use these tools.

In [ ]:
# YOUR CHANGES HERE

tools = [
    {
        "type": "function",
        "name": "describe_database",
        "description": (
            "List the available DuckDB tables or describe the "
            "columns and types of a specified table. "
            "Use this tool to discover the database structure "
            "before writing SQL queries."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "table": {
                    "type": ["string", "null"],
                    "description": (
                        "Name of the table to describe, "
                        "or null to list all tables."
                    )
                }
            },
            "required": ["table"],
            "additionalProperties": False
        },
        "strict": True
    },
]


The functions dictionary is used to map responses requesting function invocations back to Python functions.

In [ ]:
functions = {
    "describe_database": describe_database,
    "run_sql": run_sql,
    "create_chart": create_chart,
    "create_report": create_report,
}

In [ ]:
def log_event(file, event, **data):
    """Append one JSON event to an open log file."""
    record = {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "event": event,
        **data,
    }
    file.write(json.dumps(record, default=str) + "\n")
    file.flush()

The `run_analyst` function below provides an agent loop calling the Responses API, logging the results, and invoking tools.
Most of the function is complete, but it lacks instructions to the agent to steer its behavior.
You will need to fill in the missing instructions.

In [ ]:
# YOUR CHANGES HERE

def run_analyst(
    prompt,
    model="gpt-6-luna",
    max_steps=30,
    log_filename="analyst_log.jsonl",
):
    run_id = str(uuid.uuid4())
    previous_response_id = None
    next_input = prompt

    total_tokens = 0
    total_tool_calls = 0

    with open(log_filename, "a", encoding="utf-8") as log:

        def record(event, **data):
            log_event(log, event, run_id=run_id, **data)

        record("run_start", prompt=prompt, model=model)

        for step in tqdm.notebook.tqdm(range(1, max_steps + 1)):
            # Call the model.
            start = time.perf_counter()

            try:
                kwargs = {}
                if previous_response_id is not None:
                    kwargs["previous_response_id"] = previous_response_id

                response = client.responses.create(
                    model=model,
                    input=next_input,
                    tools=tools,
                    instructions=(
                        "YOUR CHANGES HERE"
                    ),
                    **kwargs,
                )
            except Exception as e:
                record("api_error", step=step, error=str(e))
                raise

            elapsed = time.perf_counter() - start

            usage = response.usage
            if usage is not None:
                total_tokens += usage.total_tokens

            record(
                "model_response",
                step=step,
                response_id=response.id,
                status=response.status,
                elapsed_seconds=elapsed,
                input_tokens=usage.input_tokens if usage else None,
                output_tokens=usage.output_tokens if usage else None,
            )

            if response.status != "completed":
                record("run_end", status=response.status)
                raise RuntimeError(
                    f"Model response status: {response.status}"
                )

            calls = [
                item for item in response.output
                if item.type == "function_call"
            ]

            # The agent has finished.
            if not calls:
                answer = response.output_text

                record(
                    "run_end",
                    status="completed",
                    answer=answer,
                    total_tokens=total_tokens,
                    total_tool_calls=total_tool_calls,
                )

                return IPython.display.display(IPython.display.Markdown(answer))

            # Execute all requested tool calls.
            outputs = []

            for call in calls:
                total_tool_calls += 1

                record(
                    "tool_call",
                    step=step,
                    call_id=call.call_id,
                    name=call.name,
                    arguments=call.arguments,
                )

                start = time.perf_counter()

                try:
                    args = json.loads(call.arguments)
                    result = functions[call.name](**args)
                except Exception as e:
                    result = {"ok": False, "error": str(e)}

                elapsed = time.perf_counter() - start

                record(
                    "tool_result",
                    step=step,
                    call_id=call.call_id,
                    name=call.name,
                    result=result,
                    elapsed_seconds=elapsed,
                )

                outputs.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(result, default=str),
                })

            # Continue the conversation using the tool results.
            previous_response_id = response.id
            next_input = outputs

        record(
            "run_end",
            status="max_steps_exceeded",
            total_tokens=total_tokens,
            total_tool_calls=total_tool_calls,
        )

        raise RuntimeError("Agent exceeded maximum steps.")


In [ ]:
run_analyst("how many districts are there?")

# Part 2: Produce an Infrastructure Safety Report

The city manager is concerned that potentially dangerous infrastructure incidents are not always being identified or handled consistently. You have been asked to prepare an evidence-based report.

Use `run_analyst` to analyze the data given to identify patterns in potentially hazardous incidents.
Identify any significant concentrations, and back them up with quantitative evidence.
Generate a report `report.md` with at least 3 charts supporting your conclusions.

Your final report must include:

1. **Executive summary:** The main findings in a short paragraph.
2. **Quantitative results:** At least three specific findings, with values that can be reproduced from the data.
3. **Visualizations:** At least two charts produced using your agent's tools.
4. **Classification analysis:** At least one comparison between the original intake classifications and your LLM-derived fields.
5. **Caveats:** Explain how uncertain labels, missing data, differences in district populations, and incomplete service requests might affect the conclusions.
6. **Recommendation:** Suggest one concrete issue the city should investigate further, without presenting correlation as proof of causation.

You may interact with your agent, ask follow-up questions, and request revisions. However, the reported numbers must come from tool outputs rather than being invented by the language model.

**Make sure to check your report before submission; it will be manually graded by the teaching staff.**

In [ ]:
# YOUR CHANGES HERE

long_prompt = """YOUR CHANGES HERE"""
run_analyst(long_prompt)

# Part 3: Code

Submit your Jupyter notebook, `report.md` produced by `run_analyst`, all of its embedded images, and the report-generation transcript `report_log.jsonl` to Gradescope.

# Part 4: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.